# Lab 07 — Spectral estimator choice & the robust-peak checklist

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. The instructor solution lives in `labs/solutions/` and is not included here.

**Covers.** Chapter 7 — §7.2 (why the periodogram is inconsistent), §7.5 (Welch's method), §7.9 (model-order selection and the decision framework).

**Biomedical question.** Is this spectral peak real — and which estimator should I trust?
**Task type (§1.8).** Spectral estimation + claim discipline (tell a real rhythm from a variance / leakage artifact)
**Information that must be preserved.** the distinction between a *real narrowband rhythm* and a *variance / leakage artifact* — a peak I would report as physiology has to survive the robust-peak checklist, not merely look tall on one plot.
**Main assumptions.** the record is (locally) stationary and cleanly sampled (`fs` known, no aliasing); the raw periodogram is an *inconsistent* estimator — its per-bin variance does NOT fall with record length — while Welch trades frequency resolution for variance by averaging segments.
**Primary diagnostic.** (a) per-bin scatter of the periodogram vs Welch in a flat band; (b) a four-point robust-peak checklist — persistence across *window type*, persistence across *segment length*, *excess over the estimator's own variability*, and *not a known line frequency* — applied **identically** to a real peak and to a fake one.
**Transfer challenge.** re-run the checklist on your own recording's suspicious peak, or swap Welch for a multitaper estimate and confirm the verdict does not change.

*Self-contained: one seed (`np.random.default_rng(2013)`), `scipy.signal` only, no `bsp`, no I/O; runs offline in well under a minute.*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab07_estimator_choice_robust_peak/lab07_estimator_choice_robust_peak.ipynb) [![View](https://img.shields.io/badge/view-static-orange)](https://farhad-abtahi.github.io/CM2013/nb/lab07_estimator_choice_robust_peak.html) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab07_estimator_choice_robust_peak.ipynb)

In [ ]:
# --- shared setup (reproducible; fully offline, scipy.signal only) ---
import numpy as np, matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(2013)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

fs = 100.0                       # Hz — sampling rate for the whole lab
LINE_FREQS = (50.0, 60.0)        # mains lines we refuse to trust (EU / US)
REAL_F = 10.0                    # the one real narrowband rhythm we will defend

def coloured_noise(n, sd=1.0):
    """Smooth low-frequency background: white noise shaped by a 2nd-order low Butterworth."""
    b, a = sig.butter(2, 8.0/(fs/2), btype="low")
    c = sig.lfilter(b, a, rng.standard_normal(n))
    return c * (sd / (c.std() + 1e-12))

def biomedical_signal(n):
    """Coloured background + ONE real 10 Hz rhythm + a STRONG 3 Hz component (a leakage source).
    Nothing real lives above 12 Hz except that 10 Hz line, so any high-band 'peak' is suspect."""
    t = np.arange(n) / fs
    x  = coloured_noise(n, sd=1.0)                # smooth sloping floor
    x += 0.30 * rng.standard_normal(n)            # small broadband floor (keeps the high band sane)
    x += 1.00 * np.sin(2*np.pi*REAL_F*t + 0.7)    # the REAL rhythm to defend (10 Hz)
    x += 6.00 * np.sin(2*np.pi*3.0*t + 0.2)       # strong low-frequency component (leakage source)
    return x, t

# One long white-noise process for Task 1 (take prefixes = "same process, more samples").
NS = (512, 2048, 8192)
white_long = rng.standard_normal(max(NS))

# One biomedical record for Tasks 2-3 (~41 s; reused everywhere so the results are consistent).
N_MAIN = 4096
x_main, t_main = biomedical_signal(N_MAIN)
print(f"white-noise prefixes N = {NS};  biomedical record N = {N_MAIN} ({N_MAIN/fs:.0f} s at {fs:.0f} Hz)")


## 1. Periodogram vs Welch — the variance that will not go away
Estimate the PSD of the **same** white-noise process at three record lengths and watch what happens to the per-bin scatter. The raw periodogram is *inconsistent*: more samples buy more bins, each still ~100% uncertain. Welch averages segments and buys down the variance instead.

In [ ]:
# TODO estimate the PSD of the SAME white-noise process at N = 512, 2048, 8192 with (a) a RAW
#   periodogram (sig.periodogram, window="boxcar") and (b) Welch averaging (sig.welch,
#   window="hann", nperseg=128, noverlap=64, detrend=False), then score each in the flat band FLAT
#   with band_cv(), which returns (CV, variance) -- CV = std/mean, a scale-free per-bin scatter.
#
#   Your code must bind EXACTLY these five names inside the loop, because the lines that follow
#   the gap (the peri_cv/wel_cv bookkeeping and the table row) use them:
#       cvp, vp  <- band_cv(...) on the RAW periodogram
#       cvw, vw  <- band_cv(...) on the Welch estimate
#       K        <- the ACTUAL number of averaged Welch segments, 1 + (N - nperseg)//(nperseg - noverlap)
#   The next line, `peri_cv[N], wel_cv[N] = cvp, cvw`, is written for you -- it files your two CVs
#   for the sanity check at the end. Bind anything else you like, but bind those five or the cell
#   raises NameError on the very next statement.
#
#   What you should see: the periodogram CV stays ~1 at every N (an INCONSISTENT estimator -- more
#   samples buy more bins, each still ~100% uncertain) while the Welch CV falls as K grows.
FLAT = (10.0, 45.0)
def band_cv(f, P, lo, hi):
    m = (f >= lo) & (f <= hi)
    return float(P[m].std() / P[m].mean()), float(P[m].var())

peri_cv, wel_cv = {}, {}
print(f"{'N':>6} | {'periodogram CV':>15} {'peri var':>11} | {'Welch CV':>9} {'Welch var':>11}  (K segs)")
for N in NS:
    xw = white_long[:N]                                            # same process, first N samples
    raise NotImplementedError("TODO: implement this — see the comment above")
    peri_cv[N], wel_cv[N] = cvp, cvw
    print(f"{N:>6} | {cvp:>15.3f} {vp:>11.2e} | {cvw:>9.3f} {vw:>11.2e}  (K={K})")

# a picture: the periodogram stays grassy at every N; Welch is the smooth one
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
for N in NS:
    xw = white_long[:N]
    fp, Pp = sig.periodogram(xw, fs=fs, window="boxcar")
    fw, Pw = sig.welch(xw, fs=fs, window="hann", nperseg=128, noverlap=64, detrend=False)
    ax[0].semilogy(fp, Pp, lw=0.6, alpha=0.7, label=f"N={N}")
    ax[1].semilogy(fw, Pw, lw=1.0, alpha=0.9, label=f"N={N}")
ax[0].set_title("raw periodogram — grass never shrinks"); ax[0].set_xlabel("Hz"); ax[0].set_ylabel("PSD")
ax[1].set_title("Welch — variance falls with averaging"); ax[1].set_xlabel("Hz"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
# Checkpoint: the periodogram column of the table barely moves; the Welch column drops. Why does a
# LONGER record fail to make the periodogram smoother?


## 2. The robust-peak checklist — run it on the REAL 10 Hz rhythm
A peak worth reporting must clear four hurdles (Ch. 7). Judge it **only** with consistent (Welch) estimators, and probe it across estimator *settings* — never off a single raw plot:

| # | check | why it matters |
|---|-------|----------------|
| 1 | persists across **window type** (Hann *and* boxcar) | a leakage side-lobe is window-shaped; a real tone is not |
| 2 | persists across **segment length** (`nperseg` 256 *and* 512) | a variance spike hops with the bin grid; a real line stays put |
| 3 | **exceeds the estimator's own variability** | Welch's relative std ~ 1/√K — the peak must beat the grass |
| 4 | **not a known line frequency** (mains 50/60 Hz, DC) | necessary, *not* sufficient — a clean frequency can still be noise |

The 10 Hz rhythm we designed in should pass all four.

In [ ]:
# --- infrastructure: does a candidate frequency show a real, prominent peak in estimate (f, P)? ---
def peak_here(f, P, f0, tol=0.6, ann=(1.5, 4.0), prom_db=6.0):
    """True if there is a local max within +/-tol Hz of f0 standing >= prom_db above the LOCAL
    background (median PSD in an annulus [ann0, ann1] Hz around f0, which excludes the peak itself)."""
    near = np.abs(f - f0) <= tol
    if not near.any():
        return False, 0.0, 1e-30
    pk = float(P[near].max())
    ring = (np.abs(f - f0) > ann[0]) & (np.abs(f - f0) <= ann[1])
    bg = float(np.median(P[ring])) if ring.any() else float(np.median(P))
    return (10*np.log10(pk/(bg+1e-30)) >= prom_db), pk, bg

def robust_peak_checklist(x, fs, f0, label=""):
    """The four-point discipline of Ch. 7. Judge a candidate ONLY with consistent (Welch) estimators
    and across estimator settings — never off a single raw-periodogram plot."""
    n = x.size
    fH, PH = sig.welch(x, fs=fs, window="hann",   nperseg=256, noverlap=128, detrend="constant")
    fB, PB = sig.welch(x, fs=fs, window="boxcar", nperseg=256, noverlap=128, detrend="constant")
    fL, PL = sig.welch(x, fs=fs, window="hann",   nperseg=512, noverlap=256, detrend="constant")
    okH, pkH, bgH = peak_here(fH, PH, f0)
    okB, _,   _   = peak_here(fB, PB, f0)
    okL, _,   _   = peak_here(fL, PL, f0)

    # TODO check 1 — persists across WINDOW type (present with Hann AND boxcar). Set persist_window.
    raise NotImplementedError("TODO: implement this — see the comment above")
    # TODO check 2 — persists across SEGMENT length (present at nperseg 256 AND 512). Set persist_length.
    raise NotImplementedError("TODO: implement this — see the comment above")
    # TODO check 3 — the peak's excess over background beats the estimator's OWN variability. Welch's
    #   relative std ~ 1/sqrt(K); require (peak - background) > 5 * (1/sqrt(K)) * background. Set exceeds_var.
    raise NotImplementedError("TODO: implement this — see the comment above")
    # TODO check 4 — f0 is NOT a known line frequency (mains 50/60 Hz) or DC. Set not_line.
    raise NotImplementedError("TODO: implement this — see the comment above")

    checks = {"persist_window": persist_window, "persist_length": persist_length,
              "exceeds_variability": exceeds_var, "not_line_frequency": not_line}
    checks["PASS"] = bool(all(checks.values()))
    print(f"[{label}] candidate f0 = {f0:.2f} Hz")
    for k in ("persist_window", "persist_length", "exceeds_variability", "not_line_frequency"):
        print(f"    {k:20s}: {'PASS' if checks[k] else 'FAIL'}")
    print(f"    {'VERDICT':20s}: {'REAL peak — safe to report' if checks['PASS'] else 'ARTIFACT — do NOT report'}")
    return checks

real_checks = robust_peak_checklist(x_main, fs, REAL_F, label="real 10 Hz rhythm")
# Checkpoint: which single check would you drop last, and why is 'not a line frequency' the weakest?


## 3. Same checklist, FAKE peak — the periodogram's invention
A naive analyst reads the **raw periodogram of a short record** and circles the tallest bump in a clean band. Run the *identical* checklist on it. Because it is variance (and low-frequency leakage), not signal, it should fail persistence and the variability test — while still 'passing' the line-frequency check, which is exactly why that check alone is never enough.

In [ ]:
# TODO a naive analyst reads the RAW periodogram of a SHORT record and circles the tallest bump in
#   a clean band [15, 40] Hz. Find that frequency (f_art) on x_main[:512], then run the SAME
#   checklist on it. It should FAIL persistence / variability (it is variance + leakage, not signal).
raise NotImplementedError("TODO: implement this — see the comment above")

# it "only appears at one segment length": the tallest [15,40] Hz bump hops with the record length
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: which of the four checks does the artifact fail, and which (line-frequency) does it
# still 'pass'? Why is passing the line-frequency check NOT enough to trust a peak?


### Live sanity check
A metric you never see fire is untrustworthy. These asserts (numbers computed above) encode the lab's two claims: periodogram scatter stays ~constant across N while Welch's is clearly lower; and the real peak passes the checklist while the artifact fails it.

In [ ]:
# --- live sanity check: numbers are computed above; these asserts encode the lab's claims ---
# (1) periodogram per-bin scatter does NOT shrink with N; Welch's is clearly lower at every N.
for N in NS:
    assert 0.7 <= peri_cv[N] <= 1.4, f"periodogram CV at N={N} should be ~1, got {peri_cv[N]:.3f}"
    assert wel_cv[N] < 0.65,          f"Welch CV at N={N} should be well below 1, got {wel_cv[N]:.3f}"
    assert wel_cv[N] < 0.75 * peri_cv[N], f"Welch should cut the variance at N={N}"
assert abs(peri_cv[8192] - peri_cv[512]) < 0.25, "periodogram CV must stay ~flat across N"
assert wel_cv[8192] < wel_cv[512], "Welch variance must fall as more segments are averaged"
# (2) the real 10 Hz peak passes every check; the artifact fails persistence AND variability.
assert real_checks["PASS"] is True,  "the real 10 Hz rhythm must pass the checklist"
assert art_checks["PASS"] is False,  "the artifact must NOT pass the checklist"
assert not art_checks["persist_window"] and not art_checks["persist_length"], \
    "the artifact should fail the persistence checks"
assert not art_checks["exceeds_variability"], "the artifact must not beat the estimator's variability"
assert art_checks["not_line_frequency"] is True, \
    "the artifact still 'passes' the line-frequency check (necessary, not sufficient)"

n_art_fail = sum(not art_checks[k] for k in
                 ("persist_window", "persist_length", "exceeds_variability", "not_line_frequency"))
print("sanity check PASSED:")
print(f"  periodogram CV ~ {np.mean([peri_cv[N] for N in NS]):.2f} at every N (grassy);"
      f"  Welch CV {wel_cv[512]:.2f} -> {wel_cv[8192]:.2f} as K grows")
print(f"  real 10 Hz peak: PASS all 4 checks   |   artifact @ {f_art:.1f} Hz: FAILS "
      f"{n_art_fail}/4 (passes only line-frequency)")


## Reflection

This reflection is for your own practice — there is nothing to submit. What matters is the *reasoning*, not hitting a particular number.

1. **Stable vs changed.** Which verdict stayed **stable** no matter how you estimated it, and which 'peak' **changed** the moment you switched window, segment length, or estimator? Contrast the 10 Hz result with the [15, 40] Hz bump.
2. **The resolution price.** Welch is smoother than the periodogram but coarser in frequency. State the resolution (Hz per bin) you used, name one pair of real rhythms this trade could blur together, and say how you would decide the trade is acceptable for *this* question.
3. **New recording.** What evidence would convince a reviewer that the 10 Hz peak is physiology on a NEW subject / device? Name at least two checklist items and one quantity you would report **alongside** the peak frequency.

**Rule out.** Reading a tall bump off a single raw-periodogram plot as physiology — or expecting a *longer record* to average the periodogram smooth — is ruled out: the periodogram is an **inconsistent** estimator (per-bin variance ~ constant in N; you measured CV ~ 1 at every N), so a longer record buys *resolution, not lower variance*, and an un-averaged spike need not be a rhythm at all. It breaks the **§1.8** requirement to preserve the distinction between a real rhythm and a variance / leakage artifact; the disciplined estimate is Welch (or multitaper) averaging **plus** the four-point persistence checklist.

> *Your answers here.*

---
*Type-2 lab for **Biomedical Signal Processing & Data Analytics**. Synthetic signal; illustrative numbers. The lesson is the method, not the exact Hz: estimate with a consistent estimator, then make every reported peak survive the checklist.*